In [0]:
%pip install xgboost==3.2.0

In [0]:
%pip install tensorflow

In [0]:
# Importing Libraries

import os
import numpy as np
import pandas as pd
import xgboost as xgb
import mlflow
from pyspark.sql.functions import col
from pyspark.ml.functions import vector_to_array
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_recall_curve
)
from tensorflow import keras
from mlflow.tracking import MlflowClient


os.environ['MLFLOW_DFS_TMP'] = '/Volumes/workspace/ml_layer/mlflow_tmp'

def ensure_catalog():
    spark.sql("USE CATALOG workspace")
    spark.sql("USE DATABASE ml_layer")

ensure_catalog()

username = spark.sql("SELECT current_user()").collect()[0][0]
mlflow.set_experiment(f"/Users/{username}/combined_scoring")

# Loading data with original transaction amounts for business impact

def load_with_amounts(train_table, test_table):
    """Load test set with original amount column for business impact calculation."""
    train_df = spark.table(train_table) \
                    .withColumn("is_fraud", col("is_fraud").cast("double"))
    test_df  = spark.table(test_table) \
                    .withColumn("is_fraud", col("is_fraud").cast("double"))

    def extract(df):
        arr = df.withColumn("features_arr", vector_to_array("features"))
        pdf = arr.select("features_arr", "is_fraud").toPandas()
        X   = np.array(pdf["features_arr"].tolist())
        y   = pdf["is_fraud"].values
        # log_amount is feature 0 in transactions schema — reverse log1p
        amounts = np.expm1(X[:, 0])
        return X, y, amounts

    X_train, y_train, _              = extract(train_df)
    X_test,  y_test,  amounts_test   = extract(test_df)
    return X_train, y_train, X_test, y_test, amounts_test


print("Loading transactions data...")
X_txn_train, y_txn_train, X_txn_test, y_txn_test, amounts = load_with_amounts(
    "workspace.ml_layer.transaction_train_features",
    "workspace.ml_layer.transaction_test_features"
)

# Loading XGBOOST from MLflow Registry

client = MlflowClient()
registered_name = "workspace.ml_layer.Modelo_Fraude_XGB_Transactions"

try:
    # Checking registered model versions
    versions = client.search_model_versions(f"name='{registered_name}'")
    
    if not versions:
        raise RuntimeError(f"Model {registered_name} not found in Registry.")
        
    # Extracting latest Optina version
    latest_version = max([int(v.version) for v in versions])
    model_uri = f"models:/{registered_name}/{latest_version}"
    
    print(f"Loading XGBoost from Unity Catalog: {model_uri}")
    xgb_model = mlflow.xgboost.load_model(model_uri)
    
except Exception as e:
    raise RuntimeError(f"Loading failed for XGBOOST model. Error: {e}")

# Direct inference

xgb_scores = xgb_model.predict_proba(X_txn_test)[:, 1]
print("✅ XGBoost scores were successfully calculated.")

# Extracting xgboost threshold from mlflow

model_version_details = client.get_model_version(name=registered_name, version=str(latest_version))
xgb_run_id = model_version_details.run_id
    
xgb_run = client.get_run(xgb_run_id)
xgb_threshold = float(xgb_run.data.metrics.get("best_threshold", 0.5))
print(f"  Threshold loaded from MLflow (XGBoost): {xgb_threshold:.4f}")
  
print(f"Loading XGBoost from Unity Catalog: {model_uri}")
xgb_model = mlflow.xgboost.load_model(model_uri)


# Loading Autoencoder with Scaler from mlflow

ae_experiment_name = f"/Users/{username}/anomaly_detection"

try:
    experiment = mlflow.get_experiment_by_name(ae_experiment_name)
    if experiment is None:
         raise RuntimeError(f"Experiment {ae_experiment_name} not found.")

    # Searching for last successful run
    runs = mlflow.search_runs(
        experiment_ids=[experiment.experiment_id],
        filter_string="tags.mlflow.runName = 'Autoencoder_transactions'",
        order_by=["start_time DESC"],
        max_results=1
    )
    
    if len(runs) == 0:
        raise RuntimeError("Not runs found for autoencoder in mlflow.")

    run_id = runs.iloc[0]["run_id"]
    ae_threshold = float(runs.iloc[0]["metrics.anomaly_threshold_95"])
    print(f"  Threshold loaded from MLflow: {ae_threshold:.4f}")
    
    # Building URIs for model and scaler

    model_uri  = f"runs:/{run_id}/Autoencoder_transactions"
    scaler_uri = f"runs:/{run_id}/Scaler_transactions"
    
    print(f"Loading Autoencoder and Scaler from Run: {run_id[:8]}")
    ae_model  = mlflow.keras.load_model(model_uri)
    ae_scaler = mlflow.sklearn.load_model(scaler_uri)

except Exception as e:
    raise RuntimeError(f"Fail loading autoencoder dependencies. Error: {e}")

# Transformation and direct inference

X_txn_test_sc = ae_scaler.transform(X_txn_test)

# Rebuilding and calculating MSE

reconstructed = ae_model.predict(X_txn_test_sc, verbose=0)
ae_scores     = np.mean((X_txn_test_sc - reconstructed) ** 2, axis=1)

print("Autoencoder scores were successfully calculated.")


# Combined score

# Loading xgboost
xgb_norm = xgb_scores

# Autoencoder normalization with 95% threshold loaded from mlflow
# Limit to 1.0 to have consistent results in calculation

ae_norm = ae_scores / ae_threshold
ae_norm = np.clip(ae_norm, 0.0, 1.0)

# Business weight
WEIGHT_XGB = 0.75   # primary scorer — best PR-AUC
WEIGHT_AE  = 0.25   # secondary signal — catches novel patterns

combined_scores = (WEIGHT_XGB * xgb_norm) + (WEIGHT_AE * ae_norm)

print("✅ Combined scores were successfully calculated (Scaled between 0.0 - 1.0)")


# Complementary analysis

xgb_caught = (xgb_scores >= xgb_threshold) & (y_txn_test == 1)
ae_caught  = (ae_scores >= ae_threshold)   & (y_txn_test == 1)

both          = (xgb_caught & ae_caught).sum()
only_xgb      = (xgb_caught & ~ae_caught).sum()
only_ae       = (~xgb_caught & ae_caught).sum()
neither       = (~xgb_caught & ~ae_caught & (y_txn_test == 1)).sum()
total_fraud   = int(y_txn_test.sum())

print("\n" + "="*60)
print("  COMPLEMENTARITY ANALYSIS — XGBoost vs Autoencoder")
print("="*60)
print(f"  Total fraud cases:    {total_fraud:,}")
print(f"  Caught by both:       {both:,} ({100*both/total_fraud:.1f}%)")
print(f"  Only XGBoost caught:  {only_xgb:,} ({100*only_xgb/total_fraud:.1f}%)")
print(f"  Only Autoencoder:     {only_ae:,} ({100*only_ae/total_fraud:.1f}%) ← AE unique value")
print(f"  Missed by both:       {neither:,} ({100*neither/total_fraud:.1f}%)")

ae_unique_pct = 100 * only_ae / total_fraud
if ae_unique_pct >= 5:
    ae_recommendation = f"✅ KEEP AE — adds {ae_unique_pct:.1f}% unique fraud coverage"
elif ae_unique_pct >= 2:
    ae_recommendation = f"⚠️  AE in shadow mode — {ae_unique_pct:.1f}% marginal value"
else:
    ae_recommendation = f"❌ DROP AE — only {ae_unique_pct:.1f}% unique fraud catch"

print(f"\n  Recommendation: {ae_recommendation}")

# Risk Tier Assessment

def assign_risk_tier(score):
    if score >= 0.80:
        return "CRITICAL"
    elif score >= 0.50:
        return "HIGH"
    elif score >= 0.20:
        return "MEDIUM"
    else:
        return "LOW"

risk_tiers = pd.Series([assign_risk_tier(s) for s in combined_scores])

action_map = {
    "CRITICAL": "AUTO-BLOCK — freeze immediately, alert customer",
    "HIGH"    : "MANUAL REVIEW — fraud analyst reviews within 1 hour",
    "MEDIUM"  : "ENHANCED MONITORING — flag for next-day batch review",
    "LOW"     : "APPROVE — proceed with standard processing"
}

# Business Impact qualification

REALISTIC_MAX_TRANSACTION = 5_000  # $5,000 cap per transaction to a realistic view

amounts_capped = np.clip(amounts, 0, REALISTIC_MAX_TRANSACTION)

print(f"  Original total transaction value: ${amounts.sum():,.2f}")
print(f"  Capped total transaction value:   ${amounts_capped.sum():,.2f}")
print(f"  Transactions affected by cap:     {(amounts > REALISTIC_MAX_TRANSACTION).sum():,}")
print()

total_fraud_value     = amounts_capped[y_txn_test == 1].sum() 
total_legit_value     = amounts_capped[y_txn_test == 0].sum()
fraud_caught_xgb   = (xgb_scores >= xgb_threshold) & (y_txn_test == 1)
fraud_value_caught = amounts_capped[fraud_caught_xgb].sum()
fraud_value_missed = total_fraud_value - fraud_value_caught

# False positive cost estimation
# Industry standard: blocked legit transaction costs ~$15 in customer friction
FP_COST_PER_BLOCK = 15
false_positives   = ((xgb_scores >= xgb_threshold) & (y_txn_test == 0)).sum()
fp_cost_total     = false_positives * FP_COST_PER_BLOCK

# Recovery rate: typically only 10-20% of caught fraud is fully recovered
RECOVERY_RATE = 0.15
recovered_value = fraud_value_caught * RECOVERY_RATE

print("\n" + "="*60)
print("  BUSINESS IMPACT — ANNUALIZED PROJECTION")
print("="*60)
print(f"  Test set fraud value         : ${total_fraud_value:>15,.2f}")
print(f"  Fraud value detected         : ${fraud_value_caught:>15,.2f} ({100*fraud_value_caught/total_fraud_value:.1f}%)")
print(f"  Fraud value missed           : ${fraud_value_missed:>15,.2f}")
print(f"  Recovered value (15% rate)   : ${recovered_value:>15,.2f}")
print(f"  False positive cost          : ${fp_cost_total:>15,.2f} ({false_positives:,} blocks)")
print(f"  Net business value           : ${recovered_value - fp_cost_total:>15,.2f}")

# Annualize: test set covers ~3 months of transactions
ANNUAL_MULTIPLIER = 4
annual_recovered  = recovered_value * ANNUAL_MULTIPLIER
annual_fp_cost    = fp_cost_total * ANNUAL_MULTIPLIER
annual_net        = annual_recovered - annual_fp_cost

print(f"\n  ANNUAL PROJECTION:")
print(f"     Recovered fraud value     : ${annual_recovered:>15,.2f}")
print(f"     Customer friction cost    : ${annual_fp_cost:>15,.2f}")
print(f"     NET ANNUAL BENEFIT        : ${annual_net:>15,.2f}")

# Saving for dashboards

production_results = pd.DataFrame({
    "transaction_idx" : range(len(combined_scores)),
    "xgb_score"       : xgb_scores,
    "ae_score"        : ae_scores,
    "combined_score"  : combined_scores,
    "risk_tier"       : risk_tiers,
    "recommended_action": [action_map[t] for t in risk_tiers],
    "actual_fraud"    : y_txn_test,
    "transaction_amount": amounts_capped
})

# Risk tier summary for dashboard
tier_summary = production_results.groupby("risk_tier").agg(
    total_count      = ("actual_fraud", "count"),
    fraud_caught     = ("actual_fraud", "sum"),
    avg_combined_score = ("combined_score", "mean"),
    total_amount     = ("transaction_amount", "sum")
).reset_index()
tier_summary["fraud_rate_pct"] = (tier_summary["fraud_caught"] / tier_summary["total_count"] * 100).round(2)

# Business impact summary
business_impact = pd.DataFrame([{
    "test_period_fraud_value"     : float(total_fraud_value),
    "fraud_value_caught"          : float(fraud_value_caught),
    "fraud_value_missed"          : float(fraud_value_missed),
    "detection_rate_by_value"     : float(fraud_value_caught / total_fraud_value),
    "false_positives_count"       : int(false_positives),
    "false_positive_cost"         : float(fp_cost_total),
    "annual_recovered_projection" : float(annual_recovered),
    "annual_fp_cost_projection"   : float(annual_fp_cost),
    "annual_net_benefit"          : float(annual_net),
    "ae_unique_fraud_pct"         : float(ae_unique_pct),
    "ae_production_recommendation": ae_recommendation
}])

# Save all to Delta
spark.createDataFrame(tier_summary).write.format("delta") \
    .mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("workspace.ml_layer.production_risk_tiers")

spark.createDataFrame(business_impact).write.format("delta") \
    .mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("workspace.ml_layer.production_business_impact")

print("\nProduction scoring complete — results saved to Delta")

In [0]:
# Overview of fraud captured by Autoencoder  and XGBoost

import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 8))
real_fraud = production_results[production_results['actual_fraud'] == 1]

sns.scatterplot(
    data=real_fraud, 
    x='xgb_score', 
    y='ae_score', 
    alpha=0.5, 
    color='red'
)

# Plotting
plt.axvline(x=xgb_threshold, color='blue', linestyle='--', label=f'XGBoost threshold ({xgb_threshold:.2f})')
plt.axhline(y=ae_threshold, color='green', linestyle='--', label=f'Autoencoder threshold 95% ({ae_threshold:.2f})')

plt.title("Fraud Detection Distribution: XGBoost vs Autoencoder")
plt.xlabel("XGBoost Probability (Historical patterns)")
plt.ylabel("Anomaly Detection - Autoencoder (New Patterns)")
plt.legend()
plt.show()